List of uniprotKB taxonomy IDs

4751 (fungi)
33208 (metazoa)
2 (baccteria)
33090 (plants)

List of INTERPRO accessions to use

IPR036625E3-binding
IPR0505372-oxoacid_dehydrogenase
IPR0010782-oxoacid_DH_actylTfrase
IPR000089Biotin_lipoyl
IPR023213CAT-like_dom_sf
IPR036625E3-bd_dom_sf
IPR004167PSBD
IPR011053Single_hybrid_motif
IPR006255SucB
IPR0030162-oxoA_DH_lipoyl-BS
IPR0010782-oxoacid_DH_actylTfrase
IPR000089Biotin_lipoyl
IPR023213CAT-like_dom_sf
IPR045257E2/Pdx1
IPR036625E3-bd_dom_sf
IPR006257LAT1
IPR004167PSBD
IPR011053Single_hybrid_motif
IPR0030162-oxoA_DH_lipoyl-BS
IPR000089Biotin_lipoyl
IPR045257E2/Pdx1
IPR036625E3-bd_dom_sf
IPR004167PSBD
IPR011053Single_hybrid_motif
IPR0030162-oxoA_DH_lipoyl-BS
IPR0505372-oxoacid_dehydrogenase
IPR0010782-oxoacid_DH_actylTfrase
IPR000089Biotin_lipoyl
IPR023213CAT-like_dom_sf
IPR011053Single_hybrid_motif
IPR006255SucB


Example Interpro fetch

In [ ]:
#!/usr/bin/env python3

# standard library modules
import sys, errno, re, json, ssl
from urllib import request
from urllib.error import HTTPError
from time import sleep

BASE_URL = "https://www.ebi.ac.uk:443/interpro/api/protein/UniProt/entry/InterPro/IPR036625/taxonomy/uniprot/4751/?page_size=200&extra_fields=sequence"

HEADER_SEPARATOR = "|"
LINE_LENGTH = 80

def output_list():
  #disable SSL verification to avoid config issues
  context = ssl._create_unverified_context()

  next = BASE_URL
  last_page = False


  attempts = 0
  while next:
    try:
      req = request.Request(next, headers={"Accept": "application/json"})
      res = request.urlopen(req, context=context)
      # If the API times out due a long running query
      if res.status == 408:
        # wait just over a minute
        sleep(61)
        # then continue this loop with the same URL
        continue
      elif res.status == 204:
        #no data so leave loop
        break
      payload = json.loads(res.read().decode())
      next = payload["next"]
      attempts = 0
      if not next:
        last_page = True
    except HTTPError as e:
      if e.code == 408:
        sleep(61)
        continue
      else:
        # If there is a different HTTP error, it wil re-try 3 times before failing
        if attempts < 3:
          attempts += 1
          sleep(61)
          continue
        else:
          sys.stderr.write("LAST URL: " + next)
          raise e

    for i, item in enumerate(payload["results"]):


      entries = None
      if ("entries" in item):
        entries = item["entries"]

      if entries is not None:
        entries_header = "-".join(
          [entry["accession"] + "(" + ";".join(
            [
              ",".join(
                [ str(fragment["start"]) + "..." + str(fragment["end"])
                  for fragment in locations["fragments"]]
              ) for locations in entry["entry_protein_locations"]
            ]
          ) + ")" for entry in entries]
        )
        sys.stdout.write(">" + item["metadata"]["accession"] + HEADER_SEPARATOR
                          + entries_header + HEADER_SEPARATOR
                          + item["metadata"]["name"] + "\n")
      else:
        sys.stdout.write(">" + item["metadata"]["accession"] + HEADER_SEPARATOR + item["metadata"]["name"] + "\n")

      seq = item["extra_fields"]["sequence"]
      fastaSeqFragments = [seq[0+i:LINE_LENGTH+i] for i in range(0, len(seq), LINE_LENGTH)]
      for fastaSeqFragment in fastaSeqFragments:
        sys.stdout.write(fastaSeqFragment + "\n")

    # Don't overload the server, give it time before asking for more
    if next:
      sleep(1)

if __name__ == "__main__":
  output_list()


List of PFAM accession to use:

PF00198 2-oxoacid_dh
PF00364 Biotin_lipoyl
PF02817 E3_binding


Example PFAM fetch

In [ ]:
#!/usr/bin/env python3

# standard library modules
import sys, errno, re, json, ssl
from urllib import request
from urllib.error import HTTPError
from time import sleep

BASE_URL = "https://www.ebi.ac.uk:443/interpro/api/protein/UniProt/entry/all/pfam/PF02817/taxonomy/uniprot/4751/?page_size=200"

def output_list():
  #disable SSL verification to avoid config issues
  context = ssl._create_unverified_context()

  next = BASE_URL
  last_page = False


  #json header
  sys.stdout.write("{ \"results\": [\n")

  attempts = 0
  while next:
    try:
      req = request.Request(next, headers={"Accept": "application/json"})
      res = request.urlopen(req, context=context)
      # If the API times out due a long running query
      if res.status == 408:
        # wait just over a minute
        sleep(61)
        # then continue this loop with the same URL
        continue
      elif res.status == 204:
        #no data so leave loop
        break
      payload = json.loads(res.read().decode())
      next = payload["next"]
      attempts = 0
      if not next:
        last_page = True
    except HTTPError as e:
      if e.code == 408:
        sleep(61)
        continue
      else:
        # If there is a different HTTP error, it wil re-try 3 times before failing
        if attempts < 3:
          attempts += 1
          sleep(61)
          continue
        else:
          sys.stderr.write("LAST URL: " + next)
          raise e

    for i, item in enumerate(payload["results"]):

      sys.stdout.write(json.dumps(item))
      # for indented output replace the above line with the following
      # sys.stdout.write(json.dumps(item, indent=4))
      # for 1 record per line uncomment the following line
      # sys.stdout.write("\n")

      if last_page and i+1 == len(payload["results"]):
        sys.stdout.write("")
      else:
        sys.stdout.write(",\n")

    # Don't overload the server, give it time before asking for more
    if next:
      sleep(1)

  #json footer
  sys.stdout.write("\n] }\n")


if __name__ == "__main__":
  output_list()
